# System compatibility check

In [2]:
import sys
import ultralytics
from ultralytics.utils import checks

print(sys.version)
print(ultralytics.__version__)
print(ultralytics.__file__)
print(checks.__file__)
print(hasattr(checks, "IS_PYTHON_MINIMUM_3_13"))

3.10.20 (main, Mar 11 2026, 17:46:40) [GCC 14.3.0]
8.4.60
/home/mdptlab/miniconda3/envs/torch-gpu/lib/python3.10/site-packages/ultralytics/__init__.py
/home/mdptlab/miniconda3/envs/torch-gpu/lib/python3.10/site-packages/ultralytics/utils/checks.py
True


# Make sure we are in the correct directory and GPU is available

In [3]:
import os
os.chdir("/home/mdptlab/roaddamagedetector/yolov11")

In [4]:
os.getcwd()

'/home/pham/Documents/SHSU/Researching/roaddamagedetector/yolov11'

In [5]:
import torch
torch.cuda.is_available()

True

# Copy YOLO11s to current location


In [6]:
# Make a copy of yolo11.yaml and edit it
from pathlib import Path
import shutil
import ultralytics

ultralytics_dir = Path(ultralytics.__file__).parent
src = ultralytics_dir / "cfg" / "models" / "11" / "yolo11.yaml"

dst = Path.cwd() / "yolo11.yaml" 

shutil.copy(src, dst)

print("Copied to:", dst)

Copied to: /home/pham/Documents/SHSU/Researching/roaddamagedetector/yolov11/yolo11.yaml


# Add CoordAtt to the configuration file

In [21]:
# Make a copy of yolo11.yaml and edit it
from pathlib import Path
import shutil
import ultralytics

ultralytics_dir = Path(ultralytics.__file__).parent
src = ultralytics_dir / "cfg" / "models" / "11" / "yolo11.yaml"

dst = Path.cwd() / "yolo11s_coordatt.yaml" #should be yolo11s_cap3p4.yaml

shutil.copy(src, dst)

print("Copied to:", dst)

Copied to: /home/pham/Documents/SHSU/Researching/roaddamagedetector/yolov11/yolo11s_coordatt.yaml


In [22]:
yolo11_yaml = Path(src)
print(Path(yolo11_yaml).read_text())

# Ultralytics 🚀 AGPL-3.0 License - https://ultralytics.com/license

# Ultralytics YOLO11 object detection model with P3/8 - P5/32 outputs
# Model docs: https://docs.ultralytics.com/models/yolo11
# Task docs: https://docs.ultralytics.com/tasks/detect

# Parameters
nc: 80 # number of classes
scales: # model compound scaling constants, i.e. 'model=yolo11n.yaml' will call yolo11.yaml with scale 'n'
  # [depth, width, max_channels]
  n: [0.50, 0.25, 1024] # summary: 181 layers, 2624080 parameters, 2624064 gradients, 6.6 GFLOPs
  s: [0.50, 0.50, 1024] # summary: 181 layers, 9458752 parameters, 9458736 gradients, 21.7 GFLOPs
  m: [0.50, 1.00, 512] # summary: 231 layers, 20114688 parameters, 20114672 gradients, 68.5 GFLOPs
  l: [1.00, 1.00, 512] # summary: 357 layers, 25372160 parameters, 25372144 gradients, 87.6 GFLOPs
  x: [1.00, 1.50, 512] # summary: 357 layers, 56966176 parameters, 56966160 gradients, 196.0 GFLOPs

# YOLO11n backbone
backbone:
  # [from, repeats, module, args]
  - [-1, 1, 

In [23]:
# Make sure to add the CoordAtt by to the copied file by this point
from pathlib import Path
import re

yaml_path = Path("yolo11s_coordatt.yaml")

if not yaml_path.exists():
    raise FileNotFoundError(f"Cannot find: {yaml_path.resolve()}")

text = yaml_path.read_text()

# Prevent accidental duplicate insertion
if "CoordAtt" in text:
    print("CoordAtt exists, this is a modified one")
else:
    # Add scale: s if it is not already present
    if not re.search(r"(?m)^scale\s*:", text):
        text, n = re.subn(
            r"(?m)^(  x:\s*\[.*?\].*)$",
            r"\1\nscale: s # use YOLO11s scale",
            text,
            count=1,
        )
        if n != 1:
            raise RuntimeError("Could not find the x scale line to insert 'scale: s'.")
    
    # Insert CoordAtt after P3 layer 16
    p3_old = "  - [-1, 2, C3k2, [256, False]] # 16 (P3/8-small)"
    p3_new = (
        "  - [-1, 2, C3k2, [256, False]] # 16 (P3/8-small)\n"
        "  - [-1, 1, CoordAtt, [32]] # 17 CoordAtt after P3/8-small"
    )
    
    text = text.replace(p3_old, p3_new, 1)
    
    # Insert CoordAtt after P4 layer 19
    p4_old = "  - [-1, 2, C3k2, [512, False]] # 19 (P4/16-medium)"
    p4_new = (
        "  - [-1, 2, C3k2, [512, False]] # 20 (P4/16-medium), originally 19\n"
        "  - [-1, 1, CoordAtt, [32]] # 21 CoordAtt after P4/16-medium"
    )
    
    text = text.replace(p4_old, p4_new, 1)
    
    # Update Detect input indices, Original: [16, 19, 22], [17, 21, 24]
    detect_pattern = r"(?m)^  - \[\[16,\s*19,\s*22\],\s*1,\s*Detect,\s*\[nc\]\].*$"
    detect_new = "  - [[17, 21, 24], 1, Detect, [nc]] # Detect(P3 CoordAtt, P4 CoordAtt, P5)"
    
    text, n = re.subn(detect_pattern, detect_new, text, count=1)
    
    # ---------------------------------------------------------------------
    # Save modified YAML
    # ---------------------------------------------------------------------
    yaml_path.write_text(text)
    
    print(f"Modified YAML saved to: {yaml_path.resolve()}")

Modified YAML saved to: /home/pham/Documents/SHSU/Researching/roaddamagedetector/yolov11/yolo11s_coordatt.yaml


In [24]:
print(Path("yolo11s_coordatt.yaml").read_text())

# Ultralytics 🚀 AGPL-3.0 License - https://ultralytics.com/license

# Ultralytics YOLO11 object detection model with P3/8 - P5/32 outputs
# Model docs: https://docs.ultralytics.com/models/yolo11
# Task docs: https://docs.ultralytics.com/tasks/detect

# Parameters
nc: 80 # number of classes
scales: # model compound scaling constants, i.e. 'model=yolo11n.yaml' will call yolo11.yaml with scale 'n'
  # [depth, width, max_channels]
  n: [0.50, 0.25, 1024] # summary: 181 layers, 2624080 parameters, 2624064 gradients, 6.6 GFLOPs
  s: [0.50, 0.50, 1024] # summary: 181 layers, 9458752 parameters, 9458736 gradients, 21.7 GFLOPs
  m: [0.50, 1.00, 512] # summary: 231 layers, 20114688 parameters, 20114672 gradients, 68.5 GFLOPs
  l: [1.00, 1.00, 512] # summary: 357 layers, 25372160 parameters, 25372144 gradients, 87.6 GFLOPs
  x: [1.00, 1.50, 512] # summary: 357 layers, 56966176 parameters, 56966160 gradients, 196.0 GFLOPs
scale: s # use YOLO11s scale

# YOLO11n backbone
backbone:
  # [from, repeat

# Add the CoordAtt class
## Note: Only need to do this once (done so don't do anymore)

In [11]:
import ultralytics
from pathlib import Path

ultralytics_dir = Path(ultralytics.__file__).parent
print(ultralytics_dir)

/home/mdptlab/miniconda3/envs/torch-gpu/lib/python3.10/site-packages/ultralytics


## 1. Add CoordAtt definition

Find nn/modules/block.py under that folder and add the following class to it:
```
class CoordAtt(nn.Module):
    """
    Coordinate Attention block.

    This version preserves the number of channels:
        input:  [B, C, H, W]
        output: [B, C, H, W]
    """

    def __init__(self, c1, reduction=32):
        super().__init__()

        mip = max(8, c1 // reduction)

        self.conv1 = nn.Conv2d(c1, mip, kernel_size=1, stride=1, padding=0)
        self.bn1 = nn.BatchNorm2d(mip)
        self.act = nn.Hardswish(inplace=True)

        self.conv_h = nn.Conv2d(mip, c1, kernel_size=1, stride=1, padding=0)
        self.conv_w = nn.Conv2d(mip, c1, kernel_size=1, stride=1, padding=0)

    def forward(self, x):
        identity = x
        n, c, h, w = x.size()

        # More export-friendly than AdaptiveAvgPool2d((None, 1))
        x_h = x.mean(dim=3, keepdim=True)                      # [B, C, H, 1]
        x_w = x.mean(dim=2, keepdim=True).permute(0, 1, 3, 2)   # [B, C, W, 1]

        y = torch.cat([x_h, x_w], dim=2)                        # [B, C, H+W, 1]
        y = self.act(self.bn1(self.conv1(y)))

        x_h, x_w = torch.split(y, [h, w], dim=2)
        x_w = x_w.permute(0, 1, 3, 2)

        a_h = self.conv_h(x_h).sigmoid()                        # [B, C, H, 1]
        a_w = self.conv_w(x_w).sigmoid()                        # [B, C, 1, W]

        return identity * a_h * a_w
```

In [25]:
!gedit $ultralytics_dir/nn/modules/block.py

## 2. Expose `CoordAtt` in `__init__.py`
Look for `from .block import (` and add `CoordAtt` at the end

In [26]:
!gedit $ultralytics_dir/nn/modules/__init__.py

## 3. Import CoordAtt in tasks.py
Run the following then find the large import section like `from ultralytics.nn.modules import ()` ..., and add CoordAtt

In [18]:
!gedit $ultralytics_dir/nn/tasks.py

## 4. Add `CoordAtt` handling inside `parse_model()`
Open `nn/tasks.py`, look for `def parse_model(d, ch, verbose=True)`: and add this case
```
elif m is CoordAtt:
    c1 = ch[f]
    args = [c1, *args]
    c2 = c1
```

In [27]:
!gedit $ultralytics_dir/nn/tasks.py

# All done, need to restart Jupyter Kernel to load new class

# Label Smoothing

In [1]:
import ultralytics
from pathlib import Path

ultralytics_dir = Path(ultralytics.__file__).parent
print(ultralytics_dir)

/home/mdptlab/miniconda3/envs/torch-gpu/lib/python3.10/site-packages/ultralytics


## Step 1: add option
Open `ultralytics/cfg/default.yaml` and add the loss hyperparameters `cls_label_smoothing: 0.0`

In [2]:
!gedit $ultralytics_dir/cfg/default.yaml

## Step 2: Modify loss function
In `__init__` from `/utils/loss.py` under the `class V8DetectionLoss`, add the following code after `self.hyp=h`:
```
self.cls_label_smoothing = float(getattr(h, "cls_label_smoothing", 0.0))
self.cls_label_smoothing = max(0.0, min(self.cls_label_smoothing, 1.0))
```

In [3]:
!gedit $ultralytics_dir/utils/loss.py

## Step 3: Smooth `target_scores` before classification loss
Find the following under `V8DetectionLoss.__call__` from `utils/loss.py`:
```
target_scores_sum = max(target_scores.sum(), 1)

# Cls loss with optional class weighting
bce_loss = self.bce(pred_scores, target_scores.to(dtype))  # (bs, num_anchors, nc)
if self.class_weights is not None:
    bce_loss *= self.class_weights
loss[1] = bce_loss.sum() / target_scores_sum  # BCE
```

Replace it with:
```
target_scores_sum = max(target_scores.sum(), 1)

# ------------------------------------------------------------------
# Cls loss with optional label smoothing and class weighting
# ------------------------------------------------------------------
# target_scores has shape:
#   [batch_size, num_anchors, num_classes]
#
# Important:
#   - Smooth only foreground/positive assigned anchors.
#   - Keep background anchors as all zeros.
#   - Keep original target_scores for bbox/DFL weighting.
#   - Use target_scores_for_cls only for classification BCE.
# ------------------------------------------------------------------
target_scores_for_cls = target_scores.to(dtype)

eps = self.cls_label_smoothing

if eps > 0.0 and self.nc > 1:
    # Positive anchors have nonzero class target scores.
    pos_mask = target_scores_for_cls.sum(dim=-1, keepdim=True) > 0

    # YOLO's TaskAlignedAssigner may produce soft/alignment-weighted
    # target scores, not simply one-hot labels. Preserve each anchor's
    # original target strength.
    target_strength = target_scores_for_cls.sum(dim=-1, keepdim=True).clamp_min(1e-9)

    # Convert target scores into a class distribution.
    target_dist = target_scores_for_cls / target_strength

    # Apply label smoothing:
    #   positive class: 1 - eps + eps / nc
    #   other classes: eps / nc
    target_dist = target_dist * (1.0 - eps) + eps / self.nc

    # Restore original target strength.
    smoothed_scores = target_dist * target_strength

    # Apply smoothing only to positive anchors.
    # Background anchors remain all zeros.
    target_scores_for_cls = torch.where(pos_mask, smoothed_scores, target_scores_for_cls)

bce_loss = self.bce(pred_scores, target_scores_for_cls)  # (bs, num_anchors, nc)

if self.class_weights is not None:
    bce_loss *= self.class_weights

loss[1] = bce_loss.sum() / target_scores_sum  # BCE
```

In [4]:
!gedit $ultralytics_dir/utils/loss.py

In [5]:
# fix the tab and sapce indentation issue
from pathlib import Path

# Change this to your actual Ultralytics loss.py path
loss_path = Path(f"{ultralytics_dir}/utils/loss.py")

text = loss_path.read_text()

# Convert tabs to equivalent spaces while preserving indentation levels
fixed_text = text.expandtabs(8)

# Verify syntax before overwriting
compile(fixed_text, str(loss_path), "exec")

# Overwrite original loss.py
loss_path.write_text(fixed_text)

print("Done. Remaining tab characters:", fixed_text.count("\t"))

Done. Remaining tab characters: 0


## Step 4. Quick verification - Make sure to restart the kernel
Restart the Kernel (just the kernel, no need the whole Jupyter Notebook) and run the following code

In [6]:
# make sure that the label_smoothing option is available
from ultralytics.cfg import DEFAULT_CFG_DICT
print(DEFAULT_CFG_DICT.get("label_smoothing", "not found"))

0.0


# Confusion Pair Label Smoothing


In [1]:
import ultralytics
from pathlib import Path

ultralytics_dir = Path(ultralytics.__file__).parent
print(ultralytics_dir)

/home/mdptlab/miniconda3/envs/torch-gpu/lib/python3.10/site-packages/ultralytics


## Step 1: add option
Open `ultralytics/cfg/default.yaml` and add the loss hyperparameters `cp_label_smoothing: 0.0` and `label_smoothing_mode: cls`

In [11]:
!gedit $ultralytics_dir/cfg/default.yaml

## Step 2: Modify loss function
In `__init__` from `/utils/loss.py` under the `class V8DetectionLoss`, add the following code after `self.hyp=h`:
```
self.cp_label_smoothing = float(getattr(h, "cp_label_smoothing", 0.0))
self.cp_label_smoothing = max(0.0, min(self.cp_label_smoothing, 1.0))
# Default: regular class-wide label smoothing
self.label_smoothing_mode = getattr(self.hyp, "label_smoothing_mode", "cls")
```

In [12]:
!gedit $ultralytics_dir/utils/loss.py

## Step 3: Smooth `target_scores` before classification loss
Find the following under `V8DetectionLoss.__call__` from `utils/loss.py`:
```
target_scores_sum = max(target_scores.sum(), 1)

# Cls loss with optional class weighting
bce_loss = self.bce(pred_scores, target_scores.to(dtype))  # (bs, num_anchors, nc)
if self.class_weights is not None:
    bce_loss *= self.class_weights
loss[1] = bce_loss.sum() / target_scores_sum  # BCE
```


```
target_scores_sum = max(target_scores.sum(), 1)

# ------------------------------------------------------------------
# Cls loss with selectable label smoothing
# ------------------------------------------------------------------
# label_smoothing_mode:
#   "cls"  -> regular class-wide label smoothing, default
#   "cp"   -> confusion-pair label smoothing between D00 and D10 only
#   "none" -> no label smoothing
# ------------------------------------------------------------------
target_scores_for_cls = target_scores.to(dtype)

label_smoothing_mode = getattr(self, "label_smoothing_mode", "cls")

if label_smoothing_mode == "cls":
    eps = self.cls_label_smoothing

    if eps > 0.0 and self.nc > 1:
        # Positive anchors have nonzero class target scores.
        pos_mask = target_scores_for_cls.sum(dim=-1, keepdim=True) > 0

        # Preserve original TaskAlignedAssigner target strength.
        target_strength = target_scores_for_cls.sum(dim=-1, keepdim=True).clamp_min(1e-9)

        # Convert target scores into a class distribution.
        target_dist = target_scores_for_cls / target_strength

        # Regular label smoothing across all classes.
        target_dist = target_dist * (1.0 - eps) + eps / self.nc

        # Restore original target strength.
        smoothed_scores = target_dist * target_strength

        # Apply smoothing only to positive anchors.
        target_scores_for_cls = torch.where(pos_mask, smoothed_scores, target_scores_for_cls)

elif label_smoothing_mode == "cp":
    eps = self.cp_label_smoothing

    if eps > 0.0 and self.nc > 1:
        # Positive anchors have nonzero class target scores.
        pos_mask = target_scores_for_cls.sum(dim=-1, keepdim=True) > 0

        # Preserve original TaskAlignedAssigner target strength.
        target_strength = target_scores_for_cls.sum(dim=-1, keepdim=True).clamp_min(1e-9)

        # Convert target scores into a class distribution.
        target_dist = target_scores_for_cls / target_strength

        # --------------------------------------------------------------
        # Confusion-pair label smoothing: D00 <-> D10 only
        #
        # Class order:
        #   0: D00
        #   1: D10
        #   2: D20
        #   3: D40
        # --------------------------------------------------------------
        d00_idx = 0
        d10_idx = 1

        smoothed_dist = target_dist.clone()

        p_d00 = target_dist[..., d00_idx:d00_idx + 1]
        p_d10 = target_dist[..., d10_idx:d10_idx + 1]

        smoothed_dist[..., d00_idx:d00_idx + 1] = (1.0 - eps) * p_d00 + eps * p_d10
        smoothed_dist[..., d10_idx:d10_idx + 1] = (1.0 - eps) * p_d10 + eps * p_d00

        # Restore original target strength.
        smoothed_scores = smoothed_dist * target_strength

        # Apply smoothing only to positive anchors.
        target_scores_for_cls = torch.where(pos_mask, smoothed_scores, target_scores_for_cls)

elif label_smoothing_mode == "none":
    pass

else:
    raise ValueError(
        f"Unknown label_smoothing_mode='{label_smoothing_mode}'. "
        "Use 'cls', 'cp', or 'none'."
    )

bce_loss = self.bce(pred_scores, target_scores_for_cls)  # (bs, num_anchors, nc)

if self.class_weights is not None:
    bce_loss *= self.class_weights

loss[1] = bce_loss.sum() / target_scores_sum  # BCE
```

In [9]:
!gedit $ultralytics_dir/utils/loss.py

In [8]:
# fix the tab and sapce indentation issue
from pathlib import Path

# Change this to your actual Ultralytics loss.py path
loss_path = Path(f"{ultralytics_dir}/utils/loss.py")

text = loss_path.read_text()

# Convert tabs to equivalent spaces while preserving indentation levels
fixed_text = text.expandtabs(8)

# Verify syntax before overwriting
compile(fixed_text, str(loss_path), "exec")

# Overwrite original loss.py
loss_path.write_text(fixed_text)

print("Done. Remaining tab characters:", fixed_text.count("\t"))

Done. Remaining tab characters: 0


# Step 4. Quick verification - Make sure to restart the kernel

In [1]:
# make sure that the label_smoothing option is available
from ultralytics.cfg import DEFAULT_CFG_DICT
print(DEFAULT_CFG_DICT.get("cls_label_smoothing", "not found"))
print(DEFAULT_CFG_DICT.get("cp_label_smoothing", "not found"))
print(DEFAULT_CFG_DICT.get("label_smoothing_mode", "not found"))

0.0
0.0
cls
